# Integration_Harmony_all

- data post scrublet detection, merged with meta data from scanpy
- no downstream processing (eg. removal of low QC cells)
    

In [ ]:
import matplotlib.pyplot as plt
import tqdm as notebook_tqdm
from datetime import date
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import scipy
import sys
import os
import re
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import LinearSegmentedColormap
#import decoupler as dc
import decoupler as dc, omnipath as op


import palantir
import matplotlib as mpl
import logging


#import src.visualization.scBasic as kvis

f"Last execution: {date.today()}"


In [ ]:
import pandas as pd

In [ ]:
from matplotlib import rcParams

In [ ]:
print(np.__version__)
print(sc.__version__)
print(sys.version)

In [ ]:
# Suppress Scanpy messages such as:
# "WARNING: saving figure to file ..."
sc.settings.verbosity = 0

# Suppress fontTools PDF font-subsetting messages
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

## Set input and output directories

In [ ]:
# -- Set base directory
output_dir = '/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/12_humanembryo/'


pd.set_option("display.max_columns", 50)
%matplotlib inline

Since I alrady ran the first part, run load the integrated adata and skip to analysis.

In [ ]:
adata = sc.read_h5ad( output_dir  +"adata_integration_harmony.h5ad")

In [ ]:
adata.obs.to_csv(output_dir + "/meta_adata_integration_harmony.csv")

In [ ]:
#meta file 
meta = '/nfs/team292/rs40/projects/human_embryo_analysis/processed_data/meta/integration/meta_integrated.csv'

## Default scanpy settings

In [ ]:
sc.settings.figdir =output_dir 


plt.rcParams.update({
    "figure.figsize": (3, 3),    # Default figure size
    "figure.dpi": 300,           # High resolution
    "font.size": 7,              # Global font size (fallback)
    "axes.titlesize": 7,        # Title size (e.g., 'Leiden')
    "axes.labelsize": 7,         # Axis labels (e.g., 'UMAP1')
    "xtick.labelsize": 7,        # Tick numbers on X axis & Colorbars
    "ytick.labelsize": 7,        # Tick numbers on Y axis & Colorbars
    "legend.fontsize": 7,        # Legend text
    "lines.markersize": 1,       # Dot size in legends
    "axes.spines.top": False,    # Remove top border globally
    "axes.spines.right": False   # Remove right border globally
})


sc.settings.figdir = output_dir

sc.set_figure_params(
    scanpy=True,           # Use Scanpy's opinionated style defaults
    dpi=300,               # High resolution for publication
    dpi_save=300,          # Resolution for saved files
    frameon=True,         # Remove box around plots (cleaner)
    vector_friendly=False,  # usage for PDF/SVG editors (Illustrator)
    fontsize=7,            # Set the base font size (very small for 3-inch figures)
    figsize=(3, 3),        # Set default figure size
    #facecolor=None,     # Ensure background is white (not transparent)
    format='pdf'           # Default save format
)

plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7

# This is the critical setting for Adobe Illustrator
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # ships with Matplotlib

sc.set_figure_params(vector_friendly=False,
    color_map="viridis")


In [ ]:
cell_type_palette={
    "ICM":"#E48452",
    "EPI":"#D0555E",
    "ExM":"#FEB746",
    "HYPO":"#AC8CBA",
    "TE":"#B5CC54",
    "unknown":"grey",
    "CTB":"#4FA6A8", 
    "STB":"#433F93", 
    "EVT":"#4F7BB9"
                }

In [ ]:
cell_type_palette={
'8cell' : '#FCD888',
 'Morula' : '#D74426',
 'unspecified':'#676767',
 'ICM' :'#FBD8EC',
'ICM/EPI' :'#AB4642',
'Epiblast': "#E8973E",
'Post-EPI-1': "#AE3B50",
'Post-EPI-2': "#AE3B50",
'PS' :'#FBD8EC',
'ECTO' :'#FF80A9',
'Nasc MES' :'#E1529D',
'Emerg MES' :'#FF2C79',
'Adv MES' :'#D82679',
'ExM' :'#AB4979',
'HEP' :'#A777CD',
 'erythrocyte' :'#690FA0',
 'AX MES': '#AC39FF',
'ENDO' :'#00AAD6',
 'early TE' :'#CBEDB3',
'Trophoblast': '#407CBA',
 'mid TE': '#6AB735',
 'late TE' :'#49B91B',
 'polarTE': '#4B5AA3',
'CTB' :'#418A7D',
 'STB' :'#508E36',
 'STB-1' :'#508E36',
 'STB-2' :'#508E36',
 'STB-3' :'#508E36',
 'EVT':'#849D2C',
 'EVT-1':'#849D2C',
 'EVT-2':'#849D2C',
 'EVT-3':'#849D2C',
'Hypoblast': '#886DB0',
'HYPO-1': '#00AAD6',
'HYPO-2': '#00AAD6',
'YS MES':'#4349AA',
    "AME":'#4349AA',
 'nan' :'#676767',
 'low quality':"#808080",
    'PGC':"#F03ECF"
}


celltype_colors = {
    "early EPI": "#E14A5B",
    "Prelineage": "#FBD8EC",  # Assuming '#premorula' corresponds to '#FBD8EC'
    "Morula": "#D74426",
    "late EPI": "#AE3B36",
    "premorula": "#FBD8EC",
    "ECTO": "#FF80A9",
    "EPI.PrE.INT": "#E1529D",
    "AdvMes": "#AB4979",
    "Mesoderm": "#E1529D",
    "ExE_Mes": "#D82679",
    "PriS": "#A777CD",
    "YSE": "#690FA0",
    "Hypoblast": "#7B36FF",
    "early TE": "#B3D51C",
    "TE": "#47A79B",
    "late TE": "#49B91B",
    "polar TE": "#A79E33",
    "CTB": "#418A7D",
    "STB": "#508E36",
    "EVT": "#849D2C",
    "DE": "#00AAD6",
    "doublet": "#BBBBBB",
    "unknown": "#676767"
}

celltype_colors = {
    "naive EPI":    "#F8B949",  # warm yellow
    "blastoid EPI": "#E79033",  # orange
    "TE":       "#3F77C5",  # blue
    "unspecified":       "#676767",  # ggrey
}

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
grey_to_pink = LinearSegmentedColormap.from_list("grey_to_pink", ["#D3D3D3", "#B54574"])

adata.write_h5ad(output_dir+"/adata_integration_large.h5ad")

## Integration_Harmony

In [ ]:
adata = adata[adata.obs['developmental.stage'] != 'nan']

In [ ]:
adata.obs['developmental.stage']  = adata.obs['developmental.stage'].astype(float)

In [ ]:
sc.tl.umap(adata, min_dist=0.5, spread=1.0, random_state=0)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100)
}):
    QC_plot = sc.pl.umap(adata,
               color = ['pct_counts_mt','scrublet_cluster_score', 'leiden',"dataset", "developmental.stage","manual_celltype_annotation", "phase","n_genes"],
               wspace = 0.5,
               legend_fontsize = 'x-small',
                frameon=False,
                size = 10,
               ncols = 3,
              alpha = 0.80,
    #            legend_loc = 'on data',
                save="QCplot_integration"
              )

In [ ]:
adata.obs["developmental.stage"].unique()

In [ ]:
adata_original = adata.copy()

In [ ]:
# multiple stages
adata = adata[adata.obs["developmental.stage"] <= 6, :].copy()

In [ ]:
# if PCA already exists and you want to reuse it:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

### View manually curated marker genes

In [ ]:
#quick view of reporter genes
marker_genes= {
        '8 cell':{'ZSCAN4'},
    'morula':{'ZNF232'},
    'NCC': {'BIK','BAK1','HKDC1'},
     'ICM':{"LAMA4"},
     'EPI':{'KLF17','NODAL', 'PRDM14','DPPA2'},
     'PostE EPI':{'UTF1', 'FGF4', 'NODAL'},
     'PostL EPI' :{ 'IGFBP2', 'USP44', 'CD24','DNMT3B'},
    'PS':{'TBXT', 'SP5', 'HOXA1', 'EOMES','NODAL'},
    'TE':{'GATA2', 'GATA3', 'CDX2','YAP1'},
    'polar TE':{'CCR7','NR2F2', "CYP19A1", "DLX5", "MUC15"},
    'mural TE':{'ASCL2', 'NDRG1', 'GATA2'},
    'CTB':{'PEG10', 'FABP5', 'HIST1H4C', 'TPM1'},
    'STB':{'CGA', 'PRR9', 'ANXA1', 'LGALS16','ATF3'},
    'EVT':{'HLA-G','ITGA1','ITGA5','NCAM1', 'NOTCH1'},
    'early HYPO': {'GATA6','LRP2','ANXA3'},#before the hypoblast is fully specified 
    'late HYPO' :{ 'COL4A1', 'GDF6', 'RSPO3' , 'FST'}, #mature lineage 
    'HYPO':{'PDGFRA','GATA6','APOA1', 'FN1', 'S100A14', 'COL4A1', 'APOA2'},
    'anterior-like HYPO':{'CER1', 'GATA6','LAPTM4B','SFRP1', 'GSC','CER1','LEFTY1','LEFTY2'},
    'AM Ecto':{'TFAP2A', 'ISL1', 'GATA3'},
    'AM':{ 'GABRP', 'IGFBP3','IGFBP5', 'NPNT'},
    'ExM':{'VIM','SNAI2', 'COL6A1'},
    'NCC': {'BIK','BAK1','HKDC1'},
    'Stress':{'CDKN1A', 'DDIT3', 'DDIT4', 'SERTAD1', 'ATF3',  'SOCS3', 'HSPA1A', 'BTG2', 'GADD45B', 'NR4A2'},
    'TE?': {'NOTO', 'MIOX', 'FABP3', 'NR2F2', 'HAND1'}
}

In [ ]:
sc.set_figure_params(figsize=(2, 2))

sc.pl.dotplot(
    adata,
    groupby="leiden",
    var_names=marker_genes,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "_marker.pdf")

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {
    "0": "Trophoblast",
    "1": "Trophoblast",
    "2": "unspecified",
    "3": "Epiblast",
    "4": "Morula",
    "5": "Trophoblast",  
    "6": "8cell",
    "7": "Trophoblast",
    "8":"Hypoblast",
}

adata.obs["manual_celltype_annotation_integrated"] = adata.obs.leiden.map(cl_annotation)
#adata.obs["cell.state"] = adata.obs.leiden.map(cl_annotation)

In [ ]:
adata_temp = adata.copy()

In [ ]:
adata = adata[adata.obs['manual_celltype_annotation_integrated'] != "low quality"]

In [ ]:
timpoint = "day1"

sub = adata[adata.obs['timepoint'] == timpoint].copy()   # copy avoids view-of-view issues
idx = np.random.permutation(sub.n_obs)
sub = sub[idx, :]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3),
    "figure.dpi": 100,
    'axes.titlesize': 8,
    "legend.fontsize": 8,
    'axes.labelsize': 8,
}):
    ax = sc.pl.umap(adata, size=10, show=False)
    sc.pl.umap(
        sub,
        color=["Condition"],
        ax=ax,
        size=10,
        palette=condition_colors2,
        save=timpoint + "_annotation_" + ".pdf",
    )

In [ ]:
#to plot labels for one sample only 

sc.set_figure_params(vector_friendly=False)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3.2, 3), 
    "figure.dpi": (100)
}):
    sc.pl.umap(
        adata,
        color =["manual_celltype_annotation_integrated"],
        size = 10,
       palette = cell_type_palette,
        save = "_annotation_" +".pdf"
    )

In [ ]:
adata

In [ ]:
#to plot labels for one sample only 

sc.set_figure_params(vector_friendly=False,
    color_map="viridis")


with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3.3, 3), 
    "figure.dpi": (100)
}):
    sc.pl.umap(
        adata,
        color =['developmental.stage'],
        size = 10,
        save = "_devestage" +".pdf"
    )

In [ ]:
#to plot labels for one sample only 

sc.set_figure_params(vector_friendly=False,
    color_map="viridis")


with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3.3, 3), 
    "figure.dpi": (100)
}):
    sc.pl.umap(
        adata,
        color =['cell.state'],
        size = 10,
        save = "_devestage" +".pdf"
    )

In [ ]:
adata.obs['dataset'].unique()

In [ ]:

sub = adata[adata.obs['dataset'].isin(['meistermann', 'petropulos'])].copy()   # copy avoids view-of-view issues
idx = np.random.permutation(sub.n_obs)
sub = sub[idx, :]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3.2, 3),
    "figure.dpi": 100
}):
    ax = sc.pl.umap(adata, size=10, show=False)
    sc.pl.umap(
        sub,
        color=["cell.state"],
        ax=ax,
        size=10,
       # palette=condition_colors2,
        save=timpoint + "_cellstate" + ".pdf",
    )

## Marker genes based on differential expression

In [ ]:
sc.tl.rank_genes_groups(adata, 'manual_celltype_annotation_integrated', method='wilcoxon', key_added = 'wilcoxon')

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, 
                                n_genes=5, 
                                key="wilcoxon", 
                                groupby='manual_celltype_annotation_integrated', 
                                show_gene_labels=True, dendrogram=False,
                               save = "_celltype_marker.pdf")

### Visualise specific marker genes

In [ ]:

marker_genes_sub= {
    # 'EPI':{'KLF17','NODAL', 'PRDM14','DPPA2'},
     'PostE EPI':{'UTF1', 'FGF4', 'NODAL'},
     'PostL EPI' :{ 'IGFBP2', 'USP44', 'CD24','DNMT3B'},
    #'PS':{'TBXT', 'SP5', 'HOXA1', 'EOMES','NODAL'},
    'TE':{'GATA2', 'GATA3', 'CDX2','YAP1'},
   # 'polar TE':{'CCR7','NR2F2', "CYP19A1", "DLX5", "MUC15"},
    #'mural TE':{'ASCL2', 'NDRG1', 'GATA2'},
    'CTB':{'PEG10', 'FABP5', 'HIST1H4C', 'TPM1'},
    'STB':{'CGA', 'PRR9', 'ANXA1', 'LGALS16','ATF3'},
    'EVT':{'HLA-G','ITGA1','ITGA5','NCAM1', 'NOTCH1'},
    'early HYPO': {'GATA6','LRP2','ANXA3'},#before the hypoblast is fully specified 
    'late HYPO' :{ 'COL4A1', 'GDF6', 'RSPO3' , 'FST'}, #mature lineage 
    'HYPO':{'PDGFRA','GATA6','APOA1', 'FN1', 'S100A14', 'COL4A1', 'APOA2'},
    'anterior-like HYPO':{'CER1', 'GATA6','LAPTM4B','SFRP1', 'GSC','CER1','LEFTY1','LEFTY2'},
    #'Amnion':{'TFAP2A','ISL1','GATA3','MSX2','SOX2','VTCN1'},
    'ExM':{'VIM','SNAI2', 'COL6A1'},
    #'NCC': {'BIK','BAK1','HKDC1'},
    #'PGC':{'SOX17','TFAP2C','NANOS3','POU5F1','PRDM1','S100A10','TCL1A','FAM162B'},
    #'PGC Progenitors':{'TFAP2A','POU5F1'},
    'Stress':{'CDKN1A', 'DDIT3', 'DDIT4', 'SERTAD1', 'ATF3',  'SOCS3', 'HSPA1A', 'BTG2', 'GADD45B', 'NR4A2'}
}

In [ ]:
marker_genes = {
   # "Naive hPSC": {"KLF17", "DPPA5", "IL6ST", "ZNF729"},
    "ICM": {"ATG2A",  "ESRRB", "LAMA4", "CCR8", "EPHA4"},
    "Epiblast": {"POU5F1", "DPPA5", "GDF3",  "NANOG", "SOX2", "KLF17", "TDGF1"},
    #"Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    #"Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Trophectoderm": {"GATA3", "CDX2", "TFAP2C", "KRT7"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17"},
    "Post-Imp": {"GABRP", "ISL1", "APLNR", "CRABP2"},
}


In [ ]:
#quick view of reporter genes
marker_genes= {
        '8 cell':{'ZSCAN4'},
    'morula':{'DUXA','ZNF232', 'DNMT3L'},
    #'NCC': {'BIK','BAK1','HKDC1'},
     'ICM':{"ATG2A",  "ESRRB", "LAMA4", "CCR8", "EPHA4"},
     'EPI':{"POU5F1", "DPPA5", "GDF3",  "NANOG", "SOX2", "KLF17", "TDGF1"},
    # 'PostE EPI':{'UTF1', 'FGF4', 'NODAL'},
    # 'PostL EPI' :{ 'IGFBP2', 'USP44', 'CD24','DNMT3B'},
   # 'PS':{'TBXT', 'SP5', 'HOXA1', 'EOMES','NODAL'},
    'TE':{"GATA3", "CDX2", "TFAP2C", "KRT7"},
    'polar TE':{'CCR7','NR2F2', "CYP19A1", "DLX5", "MUC15"},
    #'mural TE':{'ASCL2', 'NDRG1', 'GATA2'},
    #'CTB':{'PEG10', 'FABP5', 'HIST1H4C', 'TPM1'},
   # 'STB':{'CGA', 'PRR9', 'ANXA1', 'LGALS16','ATF3'},
    #'EVT':{'HLA-G','ITGA1','ITGA5','NCAM1', 'NOTCH1'},
     'HYPO':{"PDGFRA", "GATA6", "GATA4", "SOX17"},   
}

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="manual_celltype_annotation_integrated",
    var_names=marker_genes, cmap=grey_to_pink,
    #categories_order = ["early EPI", "late EPI", "ExM", "TE", "CTB", "STB", "EVT", "early HYPO", "late HYPO",'HYPO (anterior-like)', 'low quality'],
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "_celltype_marker.pdf")



In [ ]:
marker_genes = {
   # "Naive hPSC": {"KLF17", "DPPA5", "IL6ST", "ZNF729"},
    "ICM": {"ATG2A",  "ESRRB", "LAMA4", "CCR8", "EPHA4"},
    "Epiblast": {"POU5F1", "DPPA5", "GDF3",  "NANOG", "SOX2", "KLF17", "TDGF1"},
    #"Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    #"Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Trophectoderm": {"GATA3", "CDX2", "TFAP2C", "KRT7"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17"}
   # "Post-Imp": {"GABRP", "ISL1", "APLNR", "CRABP2"},
}

marker_genes2 = {"adhesion": {"CDH1", "OCLN","CLDN1","TJP1","F11R", "GJA1"}}


In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))


adata_sub =  adata[adata.obs['developmental.stage'] == 5].copy()
adata_sub = adata_sub[adata_sub.obs['manual_celltype_annotation_integrated'].isin(["Epiblast", "Trophoblast", "unspecified", "Hypoblast"])].copy()

sc.pl.dotplot(
    adata_sub ,
    groupby="manual_celltype_annotation_integrated",
    var_names=marker_genes2,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "celltype_submarker.pdf")

In [ ]:
filtered_marker_genes = marker_genes

In [ ]:
# plt.rcParams.update({'font.size': 4})
# plt.rcParams['ytick.labelsize'] = 5

# FIGSIZE=(3,3.5)
# #rcParams['figure.figsize']=FIGSIZE
# sc.set_figure_params(scanpy=True, fontsize=9)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
}):
    # Make Axes
    # Number of needed rows and columns (based on the row with the most columns)
    nrow=len(filtered_marker_genes)
    ncol=max([len(vs) for vs in filtered_marker_genes.values()])
    fig,axs=plt.subplots(nrow,ncol,figsize=(2.5*ncol,2*nrow))
    # Plot expression for every marker on the corresponding Axes object
    for row_idx,(cell_type,markers) in enumerate(filtered_marker_genes.items()):
        col_idx=0
        for marker in markers:
            ax=axs[row_idx,col_idx]
            sc.pl.umap(adata,color=marker, size = 10, ax=ax,show=False,frameon=False, cmap="RdPu")
            
            # Add cell type as row label - here we simply add it as ylabel of
            # the first Axes object in the row
            if col_idx==0:
                # We disabled axis drawing in UMAP to have plots without background and border
                # so we need to re-enable axis to plot the ylabel
                ax.axis('on')
                ax.set(xlabel=None)
                ax.tick_params(
                    top='off', bottom='off', left='off', right='off', 
                    labelleft='off', labelbottom='off')
                ax.set_ylabel(cell_type+'\n', rotation=90)
                ax.set(frame_on=False)
            col_idx+=1
        # Remove unused column Axes in the current row
        while col_idx<ncol:
            axs[row_idx,col_idx].remove()
            col_idx+=1

# Alignment within the Figure
fig.tight_layout()
fig.savefig(output_dir +'/'+'umaps.pdf')

# Plot aneuploidy

In [ ]:
adata_temp=adata.copy()

In [ ]:
adata = adata_temp.copy()

In [ ]:
#some reformatting due to incompatible cell names in the adata and the ploidy file
adata.obs["cellID"] = adata.obs.index
adata.obs.index = np.where(adata.obs['dataset'] == 'ai', adata.obs["cellID"].str.split('-').str[0], adata.obs.index.map(str)).copy()
adata.obs

In [ ]:
#inferCNV  
### is this file a bit weird? seems like it messes up the plot 
inferCNV_file = '/nfs/team292/rs40/projects/scRNAseq_aneu_practice01/processed_data/inferCNV/integrated/ploidy_cell.csv'

#scploid
#scploid_file = '/nfs/team292/rs40/projects/scRNAseq_aneu_practice01/processed_data/scploid/integrated/ploidy_cell.csv'

scploid_file = '/nfs/team292/rs40/projects/scRNAseq_aneu_practice01/processed_data/scploid/integrated/ploidy_cell_embryo.csv'

In [ ]:
#scploid_df
scploid_df =  pd.read_csv(scploid_file,index_col=0,sep=",")
scploid_df["cell"] = np.where(scploid_df["dataset"] == 'ai', scploid_df["cell"].str.split('_').str[0], scploid_df["cell"].map(str)).copy()
scploid_df[scploid_df["dataset"]=="ai"]

In [ ]:
#scploid_df

scploid_df = scploid_df[['cell', 'manual_celltype_annotation','euploid_aneuploid', 'ploidy.y', 'embryo_ploidy', 'celltype','pre_or_post']]
scploid_df = scploid_df.rename(columns={'ploidy.y': "ploidy_scploid", "cell":"cell.ID", 'euploid_aneuploid' : "euploid_aneuploid_scploid"}).set_index('cell.ID')


In [ ]:
adata

In [ ]:
adata.obs.join(scploid_df, how='left', rsuffix = "_aneu",sort=False)

In [ ]:
scploid_df= scploid_df[~scploid_df.index.duplicated(keep=False)]
adata.obs = adata.obs[~adata.obs.index.duplicated(keep=False)]

In [ ]:
# adata with ploidy info
#adata.obs = adata.obs.join(inferCNV_df, how='left', rsuffix = "_aneu",sort=False).copy()
adata.obs = adata.obs.join(scploid_df, how='left', rsuffix = "_aneu",sort=False).copy()

In [ ]:
adata.obs["developmental.stage"]  = adata.obs["developmental.stage"].astype(int)

In [ ]:
#visualise the clusters so far
#plot for QC

aneu_col = { "monosomy": "#109E9D",
        "trisomy": "#F26B3B",
        'complex':"#662C91", 
        "euploid": "#C5C5C5"}

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    "savefig.dpi" :(300)
}):
    g=sc.pl.umap(adata, color=["ploidy_scploid"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 25,
           alpha = 0.75,
            palette= aneu_col, 
           wspace=0.5,
            frameon=False,
          save = "_" +"_ploidy.pdf")

g

In [ ]:
#to plot labels for one sample only 

sc.set_figure_params(vector_friendly=False,
    color_map="viridis")


with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3.3, 3), 
    "figure.dpi": (100)
}):
    sc.pl.umap(
        adata,
        color =['ploidy_scploid'],
        size = 10,
        save = "_ploidy" +".pdf"
    )

In [ ]:
#plot for QC
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100)
}):

    sc.pl.umap(adata,
               color = ["ploidy_scploid", "dataset", "cell.state", "manual_celltype_annotation_integrated", "developmental.stage",  "leiden"],
               wspace = 0.8,
               ncols = 3,
               size =10,
               alpha = 0.75,
               save = "-"+"_init_QC.pdf",
               #palette=cell_type_palette
              )


# ECAD 

In [ ]:
adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated()]

In [ ]:
genes = ["CDH1", "TJP1",  #ZO1
             "ACTB", "ACTG1",  "ITGB1", "ITGA6", "ITGAV", "ITGB5",
            "NECTIN2", "F11R" ,#JAM1
             "CDH3", "EPCAM", "DSG2","GJA1",
            "OCLN", "PVR", "NECTIN1", "BCAM", "ICAM1", "ITGA9", "CLDN1", "ITGA2", "CXADR"]

adhesion_sub = ["CDH1",  #E-CAD
               "ACTB", "ACTG1",  #F-actin components
                 "OCLN","CLDN1","TJP1","F11R",  #ZO1, tight junction related
                 "GJA1", #gap junction
                 "EPCAM"] #Epithelial Cell Adhesion Molecule

In [ ]:
adata_sub =  adata[adata.obs['developmental.stage'].isin([6])].copy()
adata_sub = adata_sub[adata_sub.obs['manual_celltype_annotation_integrated'].isin(["Epiblast", "Trophoblast", "unspecified", "Hypoblast"])].copy()
sc.pl.dotplot(adata_sub , genes ,groupby='manual_celltype_annotation_integrated', title = "day6", standard_scale = "var", save = "_manual_celltype_annotation.pdf")

In [ ]:
adata_sub =  adata[adata.obs['developmental.stage'].isin([6])].copy()
adata_sub = adata_sub[adata_sub.obs['manual_celltype_annotation_integrated'].isin(["Epiblast", "Trophoblast", "unspecified", "Hypoblast"])].copy()
sc.pl.dotplot(adata_sub , genes ,groupby='ploidy_scploid', title = "day6", standard_scale = "var", save = "D6_ploidy.pdf")



In [ ]:
#adata_sub = adata_sub[adata_sub.obs['manual_celltype_annotation_integrated'].isin(["Epiblast", "Trophoblast", "unspecified", "Hypoblast"])].copy()
sc.pl.dotplot(adata, genes ,groupby='manual_celltype_annotation_integrated', title = "all", standard_scale = "var", save = "_manual_celltype_annotation.pdf")

In [ ]:
adata_sub =  adata[adata.obs['manual_celltype_annotation_integrated'] == "Epiblast"].copy()
sc.pl.dotplot(adata_sub, genes, groupby='ploidy_scploid', standard_scale = "var", save = "Epiblast_ploidy_scploid.pdf")

In [ ]:
adata_sub =  adata[adata.obs['manual_celltype_annotation_integrated'] == "Trophoblast"].copy()
sc.pl.dotplot(adata_sub, genes, groupby='ploidy_scploid', 
              standard_scale = "var", 
              save = "Trophoblast_ploidy_scploid.pdf")

In [ ]:
genes_use = [g for g in genes if g in adata_sub.var_names]
print(genes_use)

In [ ]:
subset_label = "unspecified"   # change if needed

adata_sub = adata[adata.obs['manual_celltype_annotation_integrated'] == subset_label].copy()

sc.pl.dotplot(
    adata_sub,
    genes_use,
    groupby='ploidy_scploid',
    standard_scale="var",
    save="unspecified_ploidy_scploid.pdf"
)

In [ ]:
subset_label = "Morula"   # change if needed

adata_sub = adata[adata.obs['manual_celltype_annotation_integrated'] == subset_label].copy()
# print("Subset shape:", adata_sub.shape)

# if adata_sub.n_obs == 0:
#     raise ValueError(f"No cells found for manual_celltype_annotation_integrated == {subset_label!r}")

# if 'ploidy_scploid' not in adata_sub.obs.columns:
#     raise ValueError("Column 'ploidy_scploid' not found in adata_sub.obs")

# adata_sub = adata_sub[adata_sub.obs['ploidy_scploid'].notna()].copy()
# print("After removing NA ploidy:", adata_sub.shape)

# if adata_sub.n_obs == 0:
#     raise ValueError("No cells left after removing NA values in 'ploidy_scploid'")

# genes_use = [g for g in genes if g in adata_sub.var_names]
# print("Genes found:", len(genes_use), "/", len(genes))

# if len(genes_use) == 0:
#     raise ValueError("None of the requested genes are in adata_sub.var_names")

sc.pl.dotplot(
    adata_sub,
    genes,
    groupby='ploidy_scploid',
    standard_scale="var",
    save="Morula_ploidy_scploid.pdf"
)

adata.write_h5ad(output_dir+"adata_integration_harmony.h5ad")

In [ ]:
adata.obs.to_csv(output_dir+"meta.csv")

In [ ]:
output_dir+"meta.csv"